# Pure DeePC Reacher Case

This notebook records a reproducible **pure DeePC** case: the executed action is exactly the first input from the DeePC optimization.

No stabilizing feedback is blended into the action:

```python
feedback_blend = 0.0
feedback_fallback = False
```

Important limitation: this is a finite-horizon success case for seed 2 over 160 control steps. It is not yet a robust multi-target pure-DeePC controller, and the same configuration drifts if run much longer. The point is to isolate a case where pure DeePC itself reaches and remains near the target for the verification window.


In [ ]:
from pathlib import Path
import json
import os

os.environ.setdefault("MPLCONFIGDIR", str(Path(".matplotlib").resolve()))
Path(os.environ["MPLCONFIGDIR"]).mkdir(exist_ok=True)

import matplotlib.pyplot as plt
import numpy as np

import expert_pipeline_search as pipe

PROJECT_DIR = Path.cwd()
HANKEL_PATH = PROJECT_DIR / "stabilizing_offline_hankel.npy"
LABEL_PATH = PROJECT_DIR / "stabilizing_prediction_error_labels.npz"
RESULT_DIR = PROJECT_DIR / "pure_deepc_results"
RESULT_DIR.mkdir(exist_ok=True)


## 1. Load Offline Data And Prediction-Error Charts


In [ ]:
H = np.load(HANKEL_PATH)
labels = np.load(LABEL_PATH, allow_pickle=True)["labels"]
print("Hankel:", H.shape)
print("cluster sizes:", np.bincount(labels.astype(int), minlength=pipe.K).tolist())


## 2. Pure DeePC Configuration

This uses one prediction-error chart and one SVD subspace basis. There is no feedback, no fallback controller, and no IK reference in the objective.


In [ ]:
variant = pipe.Variant(
    name="pure_deepc_seed2_chart3_h160",
    mode="subspace",
    rank=90,
    energy=None,
    max_cols=500,
    try_top=1,
    u_limit=0.5,
    q_xy=200000.0,
    q_dq=50.0,
    q_joint=0.0,
    q_sum=0.0,
    q_terminal_xy=0.0,
    q_terminal_dq=0.0,
    q_terminal_joint=0.0,
    q_terminal_sum=0.0,
    r=0.005,
    lam_g=0.05,
    lam_ini=100000.0,
    lam_du=5.0,
    solver="CLARABEL",
    steps=160,
    warmup_scale=0.0,
    seed=2,
    chart_selection="best_predicted",
    fixed_chart=3,
    use_ik_ref=False,
    feedback_blend=0.0,
    feedback_fallback=False,
)
variant


## 3. Run Pure DeePC


In [ ]:
result_path = RESULT_DIR / "seed2_h160_control_results.npz"
summary = pipe.run_atlas_deepc(H, labels, variant, save_path=result_path)

data = np.load(result_path, allow_pickle=True)
dist = np.asarray(data["distance"], dtype=float)
u = np.asarray(data["u"], dtype=float)

summary["zero_feedback_confirmed"] = bool(variant.feedback_blend == 0.0 and not variant.feedback_fallback)
summary["u_abs_mean"] = np.mean(np.abs(u), axis=0).tolist()
summary["u_abs_max"] = np.max(np.abs(u), axis=0).tolist()
summary["zero_action_mean_last_30_seed2"] = 0.2003005740532266
summary["result_path"] = str(result_path)

(RESULT_DIR / "seed2_h160_summary.json").write_text(json.dumps(summary, indent=2))
summary


## 4. Check Against Zero Action

For the same seed and horizon, zero action does not reach the target. Its last-window distance is about `0.2003`, while pure DeePC reaches about `0.0092` mean distance over the final 30 steps.


In [ ]:
print("pure DeePC mean last 30:", float(dist[-30:].mean()))
print("pure DeePC max last 30:", float(dist[-30:].max()))
print("zero-action mean last 30:", summary["zero_action_mean_last_30_seed2"])
print("success:", bool(dist[-30:].mean() < 0.01 and dist[-30:].max() < 0.02))
print("feedback blend:", variant.feedback_blend)
print("feedback fallback:", variant.feedback_fallback)


## 5. Tracking Diagnostics


In [ ]:
y = data["y"]
pred = data["pred"]
ref = data["ref"]
steps = np.arange(len(dist))

fig, axes = plt.subplots(6, 1, figsize=(12, 14), sharex=True)
fig.suptitle("Pure DeePC tracking diagnostics, seed 2, chart 3", fontsize=16)

for ax, (name, idx) in zip(axes[:4], [("dq1", 4), ("dq2", 5), ("x_ee", 6), ("y_ee", 7)]):
    ax.plot(steps, y[:, idx], label=f"{name} meas")
    ax.plot(steps, pred[:, idx], "--", label=f"{name} pred")
    ax.plot(steps, ref[:, idx], ":", label=f"{name} target")
    ax.set_ylabel(name)
    ax.grid(True, alpha=0.3)
    ax.legend(loc="best")

axes[4].plot(steps, u[:, 0], label="u1")
axes[4].plot(steps, u[:, 1], label="u2")
axes[4].axhline(variant.u_limit, color="red", linestyle="--", alpha=0.6, label="upper limit")
axes[4].axhline(-variant.u_limit, color="red", linestyle="--", alpha=0.6, label="lower limit")
axes[4].set_ylabel("input")
axes[4].grid(True, alpha=0.3)
axes[4].legend(loc="best")

axes[5].plot(steps, dist, label="distance")
axes[5].axhline(0.01, color="tab:green", linestyle="--", linewidth=1, label="1 cm")
axes[5].axhline(0.02, color="tab:red", linestyle="--", linewidth=1, label="2 cm")
axes[5].set_ylabel("distance")
axes[5].set_xlabel("Time step")
axes[5].grid(True, alpha=0.3)
axes[5].legend(loc="best")

fig.tight_layout()
tracking_path = RESULT_DIR / "seed2_h160_tracking.png"
fig.savefig(tracking_path, dpi=180)
tracking_path


## 6. Optional MuJoCo Video

Rendering can be slow, so it is disabled by default. Set `MAKE_VIDEO=True` to generate the MuJoCo video.


In [ ]:
MAKE_VIDEO = False
if MAKE_VIDEO:
    pipe.run_atlas_deepc(
        H,
        labels,
        variant,
        save_path=RESULT_DIR / "seed2_h160_video_results.npz",
        video_path=RESULT_DIR / "seed2_h160_mujoco.mp4",
    )
    print("Saved", RESULT_DIR / "seed2_h160_mujoco.mp4")
else:
    print("Set MAKE_VIDEO=True to render the MuJoCo video.")


## Interpretation

This demonstrates that pure DeePC can work in a favorable local case. But it also confirms the earlier concern: pure hard-chart/subspace DeePC is brittle. The same style of controller does not reliably generalize across targets, and this exact seed-2 case drifts if we extend the run to 320 steps. The robust multi-target method still needs either overlap plus a small stabilizing correction, or a stronger pure-DeePC formulation with terminal/dwell constraints and better local chart coverage.
